# 🔬 AI Research Assistant with Verified Citations
### Powered by Gemini 2.5 Flash | Runs 100% in Google Colab

**What this does:**
- Ingests PDFs, Web pages, ArXiv papers, YouTube transcripts
- Hybrid retrieval: dense vector search + BM25 with RRF fusion
- Every answer is citation-verified with `[SRC_N]` tags
- Hallucination detection built in

**Before running:** Add your Gemini API key using the 🔑 Secrets panel on the left sidebar in Colab.
- Key name: `GOOGLE_API_KEY`
- Value: your `AIza...` key from https://aistudio.google.com/apikey

---

## Step 1 — Install packages

In [ ]:
!pip install -q google-genai chromadb rank-bm25 pypdf arxiv \
    youtube-transcript-api==0.6.2 beautifulsoup4 tiktoken rich requests

print("✅ All packages installed!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 37.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━

## Step 2 — Load API key & connect to Gemini

> **How to add your key:**
> 1. Click the 🔑 **Secrets** icon in the left sidebar
> 2. Click **+ Add new secret**
> 3. Name: `GOOGLE_API_KEY` | Value: your `AIza...` key
> 4. Toggle **Notebook access** to ON
> 5. Run this cell

In [ ]:
from google.colab import userdata
import os

# Load key securely from Colab Secrets
GOOGLE_API_KEY = userdata.get('GOOGLE_API_KEY')
os.environ['GOOGLE_API_KEY'] = GOOGLE_API_KEY

# Settings
GEMINI_MODEL           = "gemini-2.5-flash"
EMBED_MODEL            = "models/gemini-embedding-001"
CHUNK_SIZE             = 512
CHUNK_OVERLAP          = 64
TOP_K_DENSE            = 8
TOP_K_SPARSE           = 8
TOP_K_FINAL            = 5
FAITHFULNESS_THRESHOLD = 0.75
CHROMA_DIR             = "/content/chroma_db"

# Test Gemini connection
from google import genai
gemini_client = genai.Client(api_key=GOOGLE_API_KEY)
test = gemini_client.models.generate_content(
    model=GEMINI_MODEL,
    contents="Reply with just the word: ready"
)
print(f"✅ Gemini connected: {test.text}")
print(f"   Model : {GEMINI_MODEL}")
print(f"   Embed : {EMBED_MODEL}")

✅ Gemini connected: ready
   Model : gemini-2.5-flash
   Embed : models/gemini-embedding-001


## Step 3 — Imports

In [ ]:
import re
import time
import hashlib
import warnings
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Dict, Optional, Tuple

import requests
import arxiv
from bs4 import BeautifulSoup
from youtube_transcript_api import YouTubeTranscriptApi
from pypdf import PdfReader
from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.markdown import Markdown
from rank_bm25 import BM25Okapi
import chromadb
import tiktoken

warnings.filterwarnings("ignore")
console = Console()
_enc    = tiktoken.get_encoding("cl100k_base")

print("✅ Imports complete.")

✅ Imports complete.


## Step 4 — Data models

In [ ]:
@dataclass
class SourceDocument:
    doc_id:   str
    source:   str
    title:    str
    url:      str
    content:  str
    metadata: Dict = field(default_factory=dict)

@dataclass
class Chunk:
    chunk_id:  str
    doc_id:    str
    source:    str
    title:     str
    url:       str
    text:      str
    chunk_idx: int

@dataclass
class RetrievedChunk:
    chunk:       Chunk
    rrf_score:   float
    dense_rank:  Optional[int] = None
    sparse_rank: Optional[int] = None

@dataclass
class CitedAnswer:
    question:          str
    answer:            str
    citations:         List[Dict]
    faithfulness:      float
    answer_relevance:  float
    latency_ms:        int
    has_hallucination: bool

print("✅ Data models defined.")

✅ Data models defined.


## Step 5 — Document loaders (PDF, Web, ArXiv, YouTube)

In [ ]:
def _make_doc_id(text: str) -> str:
    return hashlib.md5(text.encode()).hexdigest()[:10]

def load_pdf(path: str) -> SourceDocument:
    reader = PdfReader(path)
    text   = "\n".join(page.extract_text() or "" for page in reader.pages)
    title  = Path(path).stem
    console.print(f"  [green]✔ PDF[/green] '{title}' — {len(reader.pages)} pages")
    return SourceDocument(doc_id=_make_doc_id(path), source="pdf",
                          title=title, url=f"file://{path}", content=text)

def load_web(url: str) -> SourceDocument:
    resp  = requests.get(url, timeout=15, headers={"User-Agent": "Mozilla/5.0"})
    soup  = BeautifulSoup(resp.text, "html.parser")
    for tag in soup(["script","style","nav","footer","header"]):
        tag.decompose()
    text  = " ".join(soup.get_text(separator=" ").split())
    title = soup.title.string.strip() if soup.title else url
    console.print(f"  [cyan]✔ Web[/cyan] '{title[:60]}' — {len(text)} chars")
    return SourceDocument(doc_id=_make_doc_id(url), source="web",
                          title=title, url=url, content=text)

def load_arxiv(arxiv_id: str) -> SourceDocument:
    search = arxiv.Search(id_list=[arxiv_id])
    paper  = next(search.results())
    text   = (f"Title: {paper.title}\n\n"
              f"Authors: {', '.join(str(a) for a in paper.authors)}\n\n"
              f"Abstract:\n{paper.summary}")
    console.print(f"  [magenta]✔ ArXiv[/magenta] '{paper.title[:60]}'")
    return SourceDocument(doc_id=_make_doc_id(arxiv_id), source="arxiv",
                          title=paper.title, url=paper.entry_id, content=text)

def load_youtube(video_url: str) -> SourceDocument:
    vid_id = re.search(r"(?:v=|youtu\.be/)([\w-]{11})", video_url)
    if not vid_id:
        raise ValueError(f"Bad YouTube URL: {video_url}")
    vid_id = vid_id.group(1)
    transcript = YouTubeTranscriptApi.get_transcript(vid_id)
    text  = " ".join(seg["text"] for seg in transcript)
    console.print(f"  [yellow]✔ YouTube[/yellow] '{vid_id}' — {len(transcript)} segments")
    return SourceDocument(doc_id=_make_doc_id(vid_id), source="youtube",
                          title=f"YouTube:{vid_id}", url=video_url, content=text)

def ingest_sources(sources: List[Dict]) -> List[SourceDocument]:
    docs = []
    console.print("\n[bold]📥 Ingesting sources...[/bold]")
    for s in sources:
        try:
            if   s["type"] == "pdf":     docs.append(load_pdf(s["path"]))
            elif s["type"] == "web":     docs.append(load_web(s["url"]))
            elif s["type"] == "arxiv":   docs.append(load_arxiv(s["id"]))
            elif s["type"] == "youtube": docs.append(load_youtube(s["url"]))
        except Exception as e:
            console.print(f"  [red]✘ {s.get('type','?')} failed → {e}[/red]")
    console.print(f"[bold green]✅ {len(docs)} source(s) ingested.[/bold green]")
    return docs

print("✅ Document loaders ready.")

✅ Document loaders ready.


## Step 6 — Chunker + Gemini Vector Store + BM25 + RRF

In [ ]:
# ── Chunker ─────────────────────────────────────────────────────────────────
def chunk_document(doc: SourceDocument) -> List[Chunk]:
    words, chunks, start, idx = doc.content.split(), [], 0, 0
    while start < len(words):
        end = start
        while end < len(words) and len(_enc.encode(" ".join(words[start:end+1]))) < CHUNK_SIZE:
            end += 1
        text = " ".join(words[start:end])
        if text.strip():
            chunks.append(Chunk(chunk_id=f"{doc.doc_id}_c{idx:04d}",
                                doc_id=doc.doc_id, source=doc.source,
                                title=doc.title,   url=doc.url,
                                text=text,         chunk_idx=idx))
            idx += 1
        overlap = int(CHUNK_OVERLAP * len(words[start:end]) / max(1, CHUNK_SIZE))
        start   = max(start + 1, end - overlap)
    return chunks

def chunk_all(docs: List[SourceDocument]) -> List[Chunk]:
    all_chunks = []
    for doc in docs:
        c = chunk_document(doc)
        all_chunks.extend(c)
        console.print(f"  Chunked '{doc.title[:45]}' → {len(c)} chunks")
    console.print(f"[bold green]✅ Total chunks: {len(all_chunks)}[/bold green]")
    return all_chunks


# ── Gemini Embeddings ────────────────────────────────────────────────────────
def embed_texts(texts: List[str]) -> List[List[float]]:
    embeddings = []
    for text in texts:
        result = gemini_client.models.embed_content(
            model=EMBED_MODEL, contents=text
        )
        embeddings.append(result.embeddings[0].values)
    return embeddings


# ── Vector Store ─────────────────────────────────────────────────────────────
class VectorStore:
    def __init__(self):
        self._client = chromadb.PersistentClient(path=CHROMA_DIR)
        self._col    = self._client.get_or_create_collection(
                           name="research", metadata={"hnsw:space": "cosine"})
        console.print(f"  Vector store: {self._col.count()} existing chunks")

    def add_chunks(self, chunks: List[Chunk], batch_size: int = 10):
        existing = set(self._col.get()["ids"])
        new      = [c for c in chunks if c.chunk_id not in existing]
        if not new:
            console.print("  [yellow]All chunks already stored.[/yellow]"); return
        for i in range(0, len(new), batch_size):
            batch = new[i:i+batch_size]
            embs  = embed_texts([c.text for c in batch])
            self._col.add(
                ids        = [c.chunk_id for c in batch],
                embeddings = embs,
                documents  = [c.text for c in batch],
                metadatas  = [{"doc_id": c.doc_id, "source": c.source,
                               "title":  c.title,  "url":    c.url,
                               "chunk_idx": c.chunk_idx} for c in batch]
            )
            console.print(f"  [green]Embedded batch {i//batch_size+1} ✔[/green]")
        console.print(f"  [bold green]Added {len(new)} chunks.[/bold green]")

    def dense_search(self, query: str) -> List[Tuple[Chunk, float]]:
        q_emb   = embed_texts([query])[0]
        results = self._col.query(query_embeddings=[q_emb],
                                  n_results=min(TOP_K_DENSE, self._col.count()))
        out = []
        for cid, doc, meta, dist in zip(results["ids"][0], results["documents"][0],
                                         results["metadatas"][0], results["distances"][0]):
            c = Chunk(chunk_id=cid, doc_id=meta["doc_id"], source=meta["source"],
                      title=meta["title"], url=meta["url"],
                      text=doc, chunk_idx=meta["chunk_idx"])
            out.append((c, 1 - dist))
        return out


# ── BM25 ─────────────────────────────────────────────────────────────────────
class BM25Retriever:
    def __init__(self, chunks: List[Chunk]):
        self._chunks = chunks
        self._bm25   = BM25Okapi([c.text.lower().split() for c in chunks])

    def search(self, query: str) -> List[Tuple[Chunk, float]]:
        scores  = self._bm25.get_scores(query.lower().split())
        indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:TOP_K_SPARSE]
        return [(self._chunks[i], float(scores[i])) for i in indices]


# ── RRF Fusion ───────────────────────────────────────────────────────────────
def rrf_fusion(dense: List[Tuple[Chunk, float]],
               sparse: List[Tuple[Chunk, float]]) -> List[RetrievedChunk]:
    scores, cmap, dr, sr = {}, {}, {}, {}
    for rank, (c, _) in enumerate(dense,  1):
        scores[c.chunk_id] = scores.get(c.chunk_id, 0) + 1/(60+rank)
        cmap[c.chunk_id] = c; dr[c.chunk_id] = rank
    for rank, (c, _) in enumerate(sparse, 1):
        scores[c.chunk_id] = scores.get(c.chunk_id, 0) + 1/(60+rank)
        cmap[c.chunk_id] = c; sr[c.chunk_id] = rank
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:TOP_K_FINAL]
    return [RetrievedChunk(chunk=cmap[cid], rrf_score=s,
                           dense_rank=dr.get(cid), sparse_rank=sr.get(cid))
            for cid, s in ranked]


# ── Hybrid Retriever ─────────────────────────────────────────────────────────
class HybridRetriever:
    def __init__(self, vs: VectorStore, bm25: BM25Retriever):
        self._vs = vs; self._bm25 = bm25
    def retrieve(self, query: str) -> List[RetrievedChunk]:
        return rrf_fusion(self._vs.dense_search(query), self._bm25.search(query))

print("✅ Chunker, VectorStore, BM25, RRF all ready.")

✅ Chunker, VectorStore, BM25, RRF all ready.


## Step 7 — Citation-enforced answer generator

In [ ]:
SYSTEM_PROMPT = """
You are a rigorous research assistant. Answer ONLY using the provided context chunks.
Every factual claim MUST end with a citation tag [SRC_N] where N is the chunk number.
Rules:
1. Never include facts not in the chunks.
2. Every factual sentence must have at least one [SRC_N] tag.
3. Use multiple tags [SRC_1][SRC_3] when a claim has multiple sources.
4. End with a References section listing chunk titles and URLs.
""".strip()

def generate_cited_answer(question: str, retrieved: List[RetrievedChunk]) -> CitedAnswer:
    context = "\n---\n".join(
        f"[SRC_{i}] {rc.chunk.source} | {rc.chunk.title}\n"
        f"URL: {rc.chunk.url}\nText: {rc.chunk.text}"
        for i, rc in enumerate(retrieved, 1)
    )
    prompt = f"{SYSTEM_PROMPT}\n\nContext:\n{context}\n\n---\nQuestion: {question}"

    t0       = time.time()
    response = gemini_client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
    latency  = int((time.time() - t0) * 1000)
    answer   = response.text

    citations = [{"tag": f"[SRC_{i}]", "source": rc.chunk.source,
                  "title": rc.chunk.title, "url": rc.chunk.url,
                  "rrf": round(rc.rrf_score, 4)}
                 for i, rc in enumerate(retrieved, 1) if f"[SRC_{i}]" in answer]

    cited   = sum(1 for s in answer.split(".") if "[SRC_" in s)
    total   = max(1, len([s for s in answer.split(".") if len(s.strip()) > 20]))
    faith   = round(cited / total, 2)
    relev   = round(min(1.0, len(citations) / max(1, len(retrieved))), 2)

    return CitedAnswer(question=question, answer=answer, citations=citations,
                       faithfulness=faith, answer_relevance=relev,
                       latency_ms=latency, has_hallucination=(faith < FAITHFULNESS_THRESHOLD))


def display_answer(ca: CitedAnswer):
    status = "[bold red]⚠ HALLUCINATION RISK[/bold red]" if ca.has_hallucination \
             else "[bold green]✔ VERIFIED[/bold green]"
    console.print(Panel(f"[bold]Q:[/bold] {ca.question}\n\n{status}",
                        title="Research Assistant", border_style="blue"))
    console.print(Markdown(ca.answer))
    if ca.citations:
        t = Table(title="Verified Citations", show_lines=True)
        t.add_column("Tag",    style="cyan",   no_wrap=True)
        t.add_column("Source", style="green",  no_wrap=True)
        t.add_column("Title",  style="white")
        t.add_column("RRF",    style="yellow", justify="right")
        for c in ca.citations:
            t.add_row(c["tag"], c["source"], c["title"][:55], str(c["rrf"]))
        console.print(t)
    console.print(f"[dim]Faithfulness: {ca.faithfulness:.0%} · "
                  f"Relevance: {ca.answer_relevance:.0%} · Latency: {ca.latency_ms}ms[/dim]")

print("✅ Answer generator ready.")

✅ Answer generator ready.


## Step 8 — ResearchAssistant (main class)

In [ ]:
class ResearchAssistant:
    def __init__(self):
        self._docs, self._chunks, self._history = [], [], []
        self._vs = self._bm25 = self._retriever = None

    def add_sources(self, sources: List[Dict]):
        new_docs = ingest_sources(sources)
        self._docs.extend(new_docs)
        console.print("\n[bold]✂  Chunking...[/bold]")
        new_chunks = chunk_all(new_docs)
        self._chunks.extend(new_chunks)
        console.print("\n[bold]📦 Building vector store (Gemini embeddings)...[/bold]")
        if not self._vs: self._vs = VectorStore()
        self._vs.add_chunks(new_chunks)
        console.print("\n[bold]🔍 Building BM25 index...[/bold]")
        self._bm25      = BM25Retriever(self._chunks)
        self._retriever = HybridRetriever(self._vs, self._bm25)
        console.print("\n[bold green]🎉 ResearchAssistant is READY![/bold green]")

    def ask(self, question: str, show_chunks: bool = False) -> CitedAnswer:
        if not self._retriever: raise RuntimeError("Call add_sources() first.")
        console.print(f"\n[bold blue]🔎 Retrieving:[/bold blue] {question}")
        retrieved = self._retriever.retrieve(question)
        if show_chunks:
            t = Table(title="Retrieved Chunks", show_lines=True)
            t.add_column("#", style="cyan", width=3)
            t.add_column("Source", style="green", width=8)
            t.add_column("Title", width=30)
            t.add_column("RRF", style="yellow", justify="right", width=8)
            t.add_column("Preview", style="dim", width=40)
            for i, rc in enumerate(retrieved, 1):
                t.add_row(str(i), rc.chunk.source, rc.chunk.title[:28],
                          f"{rc.rrf_score:.4f}", rc.chunk.text[:60]+"…")
            console.print(t)
        console.print("[bold blue]💬 Generating cited answer...[/bold blue]")
        ca = generate_cited_answer(question, retrieved)
        self._history.append(ca)
        display_answer(ca)
        return ca

    def summary(self):
        from collections import Counter
        t = Table(title="Knowledge Base Summary", show_lines=True)
        t.add_column("Source", style="cyan")
        t.add_column("Docs",   style="yellow", justify="right")
        for src, cnt in Counter(d.source for d in self._docs).items():
            t.add_row(src, str(cnt))
        t.add_row("[bold]Total chunks[/bold]",    f"[bold]{len(self._chunks)}[/bold]")
        t.add_row("[bold]Questions answered[/bold]",f"[bold]{len(self._history)}[/bold]")
        console.print(t)

print("✅ ResearchAssistant ready.")

✅ ResearchAssistant ready.


---
## 🚀 Step 9 — RUN THE DEMO!
This ingests 5 real sources and builds your knowledge base.

In [ ]:
for m in gemini_client.models.list():
    if "embedContent" in (m.supported_actions or []):
        print(m.name)

models/gemini-embedding-001
models/gemini-embedding-2-preview


In [ ]:
EMBED_MODEL = "models/gemini-embedding-001"

In [ ]:
ra = ResearchAssistant()

SOURCES = [
    {"type": "arxiv",   "id":  "2005.11401"},  # RAG paper
    {"type": "arxiv",   "id":  "1706.03762"},  # Attention Is All You Need
    {"type": "web",     "url": "https://en.wikipedia.org/wiki/Large_language_model"},
    {"type": "web",     "url": "https://www.anthropic.com/research"},
    {"type": "youtube", "url": "https://www.youtube.com/watch?v=zjkBMFhNj_g"},
]

ra.add_sources(SOURCES)
ra.summary()

📥 Ingesting sources...

✔ ArXiv 'Retrieval-Augmented Generation for Knowledge-Intensive NLP T'

✔ ArXiv 'Attention Is All You Need'

✔ Web 'Large language model - Wikipedia' — 127093 chars

✔ Web 'Research \ Anthropic' — 3232 chars

...

Embedded batch 4 ✔

Added 35 chunks.

🔍 Building BM25 index...

🎉 ResearchAssistant is READY!

   Knowledge Base Summary    
┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━┓
┃ Source             ┃ Docs ┃
┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━┩
│ arxiv              │    2 │
├────────────────────┼──────┤
│ web                │    2 │
├────────────────────┼──────┤
│ Total chunks       │   85 │
├────────────────────┼──────┤
│ Questions answered │    0 │
└────────────────────┴──────┘

## Step 10 — Ask questions with verified citations

In [ ]:
answer1 = ra.ask(
    "How does Retrieval-Augmented Generation work and why is it better than pure LLMs?",
    show_chunks=True
)

🔎 Retrieving: How does Retrieval-Augmented Generation work and why is it better than pure LLMs?

                                            Retrieved Chunks                                             
┏━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Source   ┃ Title                          ┃      RRF ┃ Preview                                  ┃
┡━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ arxiv    │ Retrieval-Augmented Generati   │   0.0325 │ Title: Retrieval-Augmented Generation    │
│     │          │                                │          │ for Knowledge-Intensiv…                  │
├─────┼──────────┼────────────────────────────────┼──────────┼──────────────────────────────────────────┤
│ 2   │ web      │ Large language model - Wikip   │   0.0318 │ point to the deficits existing LLMs      │
│     │          │                                │          │ continue to have in pred…                │
├─────┼──────────┼────────────────────────────────┼──────────┼──────────────────────────────────────────┤
│ 3   │ web      │ Large language model - Wikip   │   0.0317 │ the base GPT-3 model can generate an     │
│     │          │                                │          │ instruction based on us…                 │
├─────┼──────────┼────────────────────────────────┼──────────┼──────────────────────────────────────────┤
│ 4   │ web      │ Large language model - Wikip   │   0.0301 │ the thought process before arriving at   │
│     │          │                                │          │ an answer. The LLM mi…                   │
├─────┼──────────┼────────────────────────────────┼──────────┼──────────────────────────────────────────┤
│ 5   │ arxiv    │ Retrieval-Augmented Generati   │   0.0161 │ tasks, outperforming parametric seq2seq  │
│     │          │                                │          │ models and task-spec…                    │
└─────┴──────────┴────────────────────────────────┴──────────┴──────────────────────────────────────────┘

💬 Generating cited answer...

╭────────────────────────────────────────────── Research Assistant ───────────────────────────────────────────────╮
│ Q: How does Retrieval-Augmented Generation work and why is it better than pure LLMs?                            │
│                                                                                                                 │
│ ✔ VERIFIED                                                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Retrieval-Augmented Generation (RAG) integrates large language models (LLMs) with document retrieval systems       
[SRC_3]. This method combines pre-trained parametric memory, which is a pre-trained seq2seq model, with            
non-parametric memory, typically a dense vector index of Wikipedia that is accessed using a pre-trained neural     
retriever [SRC_1]. When a query is given, a document retriever is activated to find the most relevant documents    
[SRC_3]. This process often involves encoding the query and documents into vectors and then locating documents     
whose vectors are most similar, usually from a vector database [SRC_3]. The LLM then produces an output by         
utilizing both the query and the context derived from the retrieved documents [SRC_3]. Two formulations of RAG     
exist: one conditions on the same retrieved passages throughout the entire generated sequence, while the other can 
employ different passages for each token [SRC_1].                                                                  

RAG models offer advantages over pure LLMs because large pre-trained language models have limitations in precisely 
manipulating and accessing knowledge, causing their performance to fall behind task-specific architectures on      
knowledge-intensive tasks [SRC_1]. Traditional LLMs also face open research problems related to providing          
provenance for their decisions and updating their world knowledge [SRC_1]. RAG models can address these issues     
through a differentiable access mechanism to explicit non-parametric memory [SRC_1]. Additionally, generative LLMs 
have been observed to "hallucinate," confidently asserting claims of fact that are not justified by their training 
data, meaning they generate syntactically sound but factually incorrect text [SRC_2]. RAG is an approach employed  
to reduce or compensate for these hallucinations [SRC_2]. RAG models have achieved state-of-the-art results on     
three open domain QA tasks, outperforming parametric seq2seq models and task-specific retrieve-and-extract         
architectures [SRC_1][SRC_5]. For language generation tasks, RAG models generate language that is more specific,   
diverse, and factual than a state-of-the-art parametric-only seq2seq baseline [SRC_1][SRC_5].                      

                                                    References                                                     

 • arxiv | Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks URL: http://arxiv.org/abs/2005.11401v4 
 • web | Large language model - Wikipedia URL: https://en.wikipedia.org/wiki/Large_language_model

                                  Verified Citations                                   
┏━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Tag     ┃ Source ┃ Title                                                   ┃    RRF ┃
┡━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ [SRC_1] │ arxiv  │ Retrieval-Augmented Generation for Knowledge-Intensive  │ 0.0325 │
├─────────┼────────┼─────────────────────────────────────────────────────────┼────────┤
│ [SRC_2] │ web    │ Large language model - Wikipedia                        │ 0.0318 │
├─────────┼────────┼─────────────────────────────────────────────────────────┼────────┤
│ [SRC_3] │ web    │ Large language model - Wikipedia                        │ 0.0317 │
├─────────┼────────┼─────────────────────────────────────────────────────────┼────────┤
│ [SRC_5] │ arxiv  │ Retrieval-Augmented Generation for Knowledge-Intensive  │ 0.0161 │
└─────────┴────────┴─────────────────────────────────────────────────────────┴────────┘

Faithfulness: 81% · Relevance: 80% · Latency: 7399ms

In [ ]:
answer2 = ra.ask(
    "What are the key innovations in the Transformer architecture?",
    show_chunks=True
)

In [ ]:
# ✏️ Change this question to anything you want!
answer3 = ra.ask("What challenges exist in AI alignment and safety?")

## Step 11 — Add your own sources anytime

In [ ]:
# Uncomment any line below to add more sources:

# ra.add_sources([
#     {"type": "arxiv",   "id":  "2310.06825"},   # Self-RAG paper
#     {"type": "web",     "url": "https://lilianweng.github.io/posts/2023-06-23-agent/"},
#     {"type": "youtube", "url": "https://www.youtube.com/watch?v=wjZofJX0v4M"},
#     {"type": "pdf",     "path": "/content/your_paper.pdf"},  # upload PDF to Colab first
# ])

# ra.ask("Your custom question here")

print("Uncomment above to add more sources and questions!")

In [ ]:
ra.add_sources([
    {"type": "pdf", "path": "/content/EBSCO-FullText-02_10_2026.pdf"},
])

ra.ask(
    "What are the factors influencing underwater Turbulence?",
    show_chunks=True
)

📥 Ingesting sources...

✔ PDF 'EBSCO-FullText-02_10_2026' — 7 pages

✅ 1 source(s) ingested.

✂  Chunking...

Chunked 'EBSCO-FullText-02_10_2026' → 15 chunks

...

╭────────────────────────────────────────────── Research Assistant ───────────────────────────────────────────────╮
│ Q: What are the factors influencing underwater Turbulence?                                                      │
│                                                                                                                 │
│ ✔ VERIFIED                                                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Optical turbulence of underwater refers to the random fluctuations and variations in the properties of light       
propagating through water [SRC_1][SRC_3]. This phenomenon is caused by small fluctuations in temperature and       
salinity in the water column [SRC_1][SRC_3]. These fluctuations lead to changes in the water's refractive index    
along the path of light [SRC_1][SRC_3]. Salinity and temperature are important factors that affect underwater      
turbulence [SRC_1]. The fluctuations of these factors greatly affect the refractive index of water, which then     
leads to changes in the optical properties of water resulting from turbulence [SRC_1].                             

                                                    References                                                     

 • [SRC_1] EBSCO-FullText-02_10_2026, file:///content/EBSCO-FullText-02_10_2026.pdf                                
 • [SRC_3] EBSCO-FullText-02_10_2026, file:///content/EBSCO-FullText-02_10_2026.pdf

                   Verified Citations                    
┏━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Tag     ┃ Source ┃ Title                     ┃    RRF ┃
┡━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ [SRC_1] │ pdf    │ EBSCO-FullText-02_10_2026 │ 0.0328 │
├─────────┼────────┼───────────────────────────┼────────┤
│ [SRC_3] │ pdf    │ EBSCO-FullText-02_10_2026 │ 0.0313 │
└─────────┴────────┴───────────────────────────┴────────┘

Faithfulness: 100% · Relevance: 40% · Latency: 4228ms

CitedAnswer(question='What are the factors influencing underwater Turbulence?', answer="Optical turbulence of underwater refers to the random fluctuations and variations in the properties of light propagating through water [SRC_1][SRC_3]. This phenomenon is caused by small fluctuations in temperature and salinity in the water column [SRC_1][SRC_3]. These fluctuations lead to changes in the water's refractive index along the path of light [SRC_1][SRC_3]. Salinity and temperature are important factors that affect underwater turbulence [SRC_1]. The fluctuations of these factors greatly affect the refractive index of water, which then leads to changes in the optical properties of water resulting from turbulence [SRC_1].\n\n### References\n*   [SRC_1] EBSCO-FullText-02_10_2026, file:///content/EBSCO-FullText-02_10_2026.pdf\n*   [SRC_3] EBSCO-FullText-02_10_2026, file:///content/EBSCO-FullText-02_10_2026.pdf", citations=[{'tag': '[SRC_1]', 'source': 'pdf', 'title': 'EBSCO-FullText-02_10_2026

In [ ]:
ra.add_sources([
    {"type": "pdf", "path": "/content/NIPS-2017-attention-is-all-you-need-Paper.pdf"},
])

📥 Ingesting sources...

✔ PDF 'NIPS-2017-attention-is-all-you-need-Paper' — 11 pages

✅ 1 source(s) ingested.

✂  Chunking...

Chunked 'NIPS-2017-attention-is-all-you-need-Paper' → 20 chunks

...

Embedded batch 1 ✔

Embedded batch 2 ✔

Added 20 chunks.

🔍 Building BM25 index...

🎉 ResearchAssistant is READY!

In [ ]:
ra.ask(
    "Applications of Attention in our Model?",
    show_chunks=True
)

🔎 Retrieving: Applications of Attention in our Model?

                                            Retrieved Chunks                                             
┏━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Source   ┃ Title                          ┃      RRF ┃ Preview                                  ┃
┡━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ pdf      │ NIPS-2017-attention-is-all-y   │   0.0328 │ the model to jointly attend to           │
│     │          │                                │          │ information from different re…           │
├─────┼──────────┼────────────────────────────────┼──────────┼──────────────────────────────────────────┤
│ 2   │ pdf      │ NIPS-2017-attention-is-all-y   │   0.0318 │ Attention" (Figure 2). The input         │
│     │          │                                │          │ consists of queries and key…             │
├─────┼──────────┼────────────────────────────────┼──────────┼──────────────────────────────────────────┤
│ 3   │ pdf      │ NIPS-2017-attention-is-all-y   │   0.0310 │ additional input when generating the     │
│     │          │                                │          │ next. The Transformer f…                 │
├─────┼──────────┼────────────────────────────────┼──────────┼──────────────────────────────────────────┤
│ 4   │ pdf      │ NIPS-2017-attention-is-all-y   │   0.0308 │ the ﬁrst Transformer models and has been │
│     │          │                                │          │ crucially involved …                     │
├─────┼──────────┼────────────────────────────────┼──────────┼──────────────────────────────────────────┤
│ 5   │ pdf      │ NIPS-2017-attention-is-all-y   │   0.0299 │ long-range dependencies in the network.  │
│     │          │                                │          │ Learning long-range …                    │
└─────┴──────────┴────────────────────────────────┴──────────┴──────────────────────────────────────────┘

💬 Generating cited answer...

╭────────────────────────────────────────────── Research Assistant ───────────────────────────────────────────────╮
│ Q: Applications of Attention in our Model?                                                                      │
│                                                                                                                 │
│ ✔ VERIFIED                                                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

The Transformer model utilizes multi-head attention in three distinct ways [SRC_1][SRC_3].                         

The first application is in "encoder-decoder attention" layers, where queries originate from the previous decoder  
layer, and the memory keys and values come from the output of the encoder [SRC_1][SRC_3]. This allows every        
position in the decoder to attend over all positions in the input sequence, mimicking typical encoder-decoder      
attention mechanisms in sequence-to-sequence models [SRC_1].                                                       

Secondly, the encoder contains self-attention layers [SRC_1][SRC_3]. In these layers, all keys, values, and queries
come from the same source, specifically the output of the previous layer in the encoder [SRC_1]. This enables each 
position in the encoder to attend to all positions in the previous layer of the encoder [SRC_1].                   

Finally, self-attention layers in the decoder allow each position in the decoder to attend to all positions up to  
and including that position within the decoder [SRC_1][SRC_3]. To maintain the auto-regressive property and prevent
leftward information flow, values in the input of the softmax corresponding to illegal connections are masked out  
[SRC_1][SRC_3].                                                                                                    

The encoder and decoder stacks are composed of stacked self-attention and point-wise, fully connected layers       
[SRC_3]. The encoder is made of N=6 identical layers, each with a multi-head self-attention mechanism and a        
position-wise fully connected feed-forward network [SRC_3]. The decoder is also composed of N=6 identical layers,  
and in addition to the two sub-layers found in the encoder, it includes a third sub-layer that performs multi-head 
attention over the output of the encoder stack [SRC_3].                                                            

                                                    References                                                     

 • NIPS-2017-attention-is-all-you-need-Paper [SRC_1] URL:                                                          
   file:///content/NIPS-2017-attention-is-all-you-need-Paper.pdf                                                   
 • NIPS-2017-attention-is-all-you-need-Paper [SRC_2] URL:                                                          
   file:///content/NIPS-2017-attention-is-all-you-need-Paper.pdf                                                   
 • NIPS-2017-attention-is-all-you-need-Paper [SRC_3] URL:                                                          
   file:///content/NIPS-2017-attention-is-all-you-need-Paper.pdf                                                   
 • NIPS-2017-attention-is-all-you-need-Paper [SRC_4] URL:                                                          
   file:///content/NIPS-2017-attention-is-all-you-need-Paper.pdf                                                   
 • NIPS-2017-attention-is-all-you-need-Paper [SRC_5] URL:                                                          
   file:///content/NIPS-2017-attention-is-all-you-need-Paper.pdf

                           Verified Citations                            
┏━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Tag     ┃ Source ┃ Title                                     ┃    RRF ┃
┡━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ [SRC_1] │ pdf    │ NIPS-2017-attention-is-all-you-need-Paper │ 0.0328 │
├─────────┼────────┼───────────────────────────────────────────┼────────┤
│ [SRC_2] │ pdf    │ NIPS-2017-attention-is-all-you-need-Paper │ 0.0318 │
├─────────┼────────┼───────────────────────────────────────────┼────────┤
│ [SRC_3] │ pdf    │ NIPS-2017-attention-is-all-you-need-Paper │  0.031 │
├─────────┼────────┼───────────────────────────────────────────┼────────┤
│ [SRC_4] │ pdf    │ NIPS-2017-attention-is-all-you-need-Paper │ 0.0308 │
├─────────┼────────┼───────────────────────────────────────────┼────────┤
│ [SRC_5] │ pdf    │ NIPS-2017-attention-is-all-you-need-Paper │ 0.0299 │
└─────────┴────────┴───────────────────────────────────────────┴────────┘

Faithfulness: 100% · Relevance: 100% · Latency: 7374ms

CitedAnswer(question='Applications of Attention in our Model?', answer='The Transformer model utilizes multi-head attention in three distinct ways [SRC_1][SRC_3].\n\nThe first application is in "encoder-decoder attention" layers, where queries originate from the previous decoder layer, and the memory keys and values come from the output of the encoder [SRC_1][SRC_3]. This allows every position in the decoder to attend over all positions in the input sequence, mimicking typical encoder-decoder attention mechanisms in sequence-to-sequence models [SRC_1].\n\nSecondly, the encoder contains self-attention layers [SRC_1][SRC_3]. In these layers, all keys, values, and queries come from the same source, specifically the output of the previous layer in the encoder [SRC_1]. This enables each position in the encoder to attend to all positions in the previous layer of the encoder [SRC_1].\n\nFinally, self-attention layers in the decoder allow each position in the decoder to attend to all positions

In [ ]:
ra.add_sources([
    {"type": "pdf", "path": "/content/MadhuSudhanKNResume-1-1.pdf"},
])

📥 Ingesting sources...

✔ PDF 'MadhuSudhanKNResume-1-1' — 1 pages

✅ 1 source(s) ingested.

✂  Chunking...

Chunked 'MadhuSudhanKNResume-1-1' → 3 chunks

...

📦 Building vector store (Gemini embeddings)...

Embedded batch 1 ✔

Added 3 chunks.

🔍 Building BM25 index...

🎉 ResearchAssistant is READY!

In [ ]:
ra.ask(
    "Summarize my skills and projects from this resume",
    show_chunks=True
)

🔎 Retrieving: Summarize my skills and projects from this resume

                                            Retrieved Chunks                                             
┏━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ #   ┃ Source   ┃ Title                          ┃      RRF ┃ Preview                                  ┃
┡━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 1   │ pdf      │ MadhuSudhanKNResume-1-1        │   0.0328 │ Madhu Sudhan K N +91 6364496493 |        │
│     │          │                                │          │ madhu.saraswathi05@gmail.c…              │
├─────┼──────────┼────────────────────────────────┼──────────┼──────────────────────────────────────────┤
│ 2   │ web      │ Large language model - Wikip   │   0.0315 │ subsequent episode. These "lessons       │
│     │          │                                │          │ learned" are stored as a …               │
├─────┼──────────┼────────────────────────────────┼──────────┼──────────────────────────────────────────┤
│ 3   │ web      │ Large language model - Wikip   │   0.0312 │ Collocation extraction Concept mining    │
│     │          │                                │          │ Coreference resolution…                  │
├─────┼──────────┼────────────────────────────────┼──────────┼──────────────────────────────────────────┤
│ 4   │ pdf      │ MadhuSudhanKNResume-1-1        │   0.0161 │ Forage), March 2025. • Participated as a │
│     │          │                                │          │ Team Leader in Stat…                     │
├─────┼──────────┼────────────────────────────────┼──────────┼──────────────────────────────────────────┤
│ 5   │ pdf      │ MadhuSudhanKNResume-1-1        │   0.0159 │ X-ray analysis and clinical metadata     │
│     │          │                                │          │ fusion.…                                 │
└─────┴──────────┴────────────────────────────────┴──────────┴──────────────────────────────────────────┘

💬 Generating cited answer...

╭────────────────────────────────────────────── Research Assistant ───────────────────────────────────────────────╮
│ Q: Summarize my skills and projects from this resume                                                            │
│                                                                                                                 │
│ ✔ VERIFIED                                                                                                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Madhu Sudhan K N possesses key skills in several areas, including programming languages, developer tools, and cloud
platforms [SRC_1]. The programming languages mentioned are Java and Python [SRC_1]. Developer tools include        
Automation Anywhere, SQL, Git, CI/CD, Docker, Kubernetes, Maven, and Jenkins [SRC_1]. For cloud platforms, Madhu   
Sudhan K N is skilled in Amazon Web Services (AWS) [SRC_1]. Additional skills encompass software development,      
automation, cybersecurity fundamentals, problem-solving, building practical applications, and analyzing system     
behavior [SRC_1]. Madhu Sudhan K N is also proficient in Open Source Intelligence (OSINT) techniques using tools   
such as Recon-ng, theHarvester, Maltego, and Shodan, and has expertise in network security tools like Nmap,        
Wireshark, Metasploit, and Burp Suite [SRC_1].                                                                     

Academic projects include the implementation of a Retrieval-Augmented Generation (RAG) pipeline in Python that uses
embedding models and vector search to improve LLM responses with document-based context [SRC_1]. Another project is
a Simple Online Bidding System utilizing blockchain technology for transparency and immutability of bids [SRC_1]. A
third project involved implementing a machine learning-based system for fraud detection in digital transactions by 
analyzing patterns and anomalies [SRC_1]. During an internship, Madhu Sudhan K N also automated OSINT Profiling of 
an Organization using reconnaissance tools like Recon-ng and theHarvester for cybersecurity assessments [SRC_1].   

                                                    References                                                     

 • MadhuSudhanKNResume-1-1 [SRC_1]                                                                                 
 • Large language model - Wikipedia [SRC_2]                                                                        
 • Large language model - Wikipedia [SRC_3]                                                                        
 • MadhuSudhanKNResume-1-1 [SRC_4]                                                                                 
 • MadhuSudhanKNResume-1-1 [SRC_5]

                       Verified Citations                       
┏━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┓
┃ Tag     ┃ Source ┃ Title                            ┃    RRF ┃
┡━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━┩
│ [SRC_1] │ pdf    │ MadhuSudhanKNResume-1-1          │ 0.0328 │
├─────────┼────────┼──────────────────────────────────┼────────┤
│ [SRC_2] │ web    │ Large language model - Wikipedia │ 0.0315 │
├─────────┼────────┼──────────────────────────────────┼────────┤
│ [SRC_3] │ web    │ Large language model - Wikipedia │ 0.0312 │
├─────────┼────────┼──────────────────────────────────┼────────┤
│ [SRC_4] │ pdf    │ MadhuSudhanKNResume-1-1          │ 0.0161 │
├─────────┼────────┼──────────────────────────────────┼────────┤
│ [SRC_5] │ pdf    │ MadhuSudhanKNResume-1-1          │ 0.0159 │
└─────────┴────────┴──────────────────────────────────┴────────┘

Faithfulness: 100% · Relevance: 100% · Latency: 4107ms

CitedAnswer(question='Summarize my skills and projects from this resume', answer='Madhu Sudhan K N possesses key skills in several areas, including programming languages, developer tools, and cloud platforms [SRC_1]. The programming languages mentioned are Java and Python [SRC_1]. Developer tools include Automation Anywhere, SQL, Git, CI/CD, Docker, Kubernetes, Maven, and Jenkins [SRC_1]. For cloud platforms, Madhu Sudhan K N is skilled in Amazon Web Services (AWS) [SRC_1]. Additional skills encompass software development, automation, cybersecurity fundamentals, problem-solving, building practical applications, and analyzing system behavior [SRC_1]. Madhu Sudhan K N is also proficient in Open Source Intelligence (OSINT) techniques using tools such as Recon-ng, theHarvester, Maltego, and Shodan, and has expertise in network security tools like Nmap, Wireshark, Metasploit, and Burp Suite [SRC_1].\n\nAcademic projects include the implementation of a Retrieval-Augmented Generation (RAG) pi